# Example how to read modify and publish content using the Confluence API


## Imports and setup

In [ ]:
import json
import requests

with open('config.json') as f:
    config = json.load(f)

assert len(config['apiurl']) > 0

authentication = (config['username'], config['password'])

(config['apiurl'], config['space'], config['page'])

In [ ]:
import xml.dom.minidom
import IPython

def beautify(flat_xml):
    dom = xml.dom.minidom.parseString('<enclosing-content>{}</enclosing-content>'.format(flat_xml))
    pretty_xml_as_string = dom.toprettyxml()
    return pretty_xml_as_string

# Read the page content

In [ ]:
resp = requests.get(config['apiurl'], auth=authentication, params= {
               'title': config['page'],
               'spaceKey': config['space'],
               'expand': 'body.view,version'
            }
        )
resp.raise_for_status()
results = resp.json().get('results')
results

Show current content with syntax higlighted HTML

In [ ]:
page = results[0]
current_content = page['body']['view']['value']
IPython.display.Code(beautify(current_content))

# Apply changes

In [ ]:
current_version = page['version']['number']
version = current_version + 1
change_message = 'Bla'.format(version)

content = current_content + '.' #current_content + '<h2>Version {}</h2><p>Modified from {}</p>'.format(version, 'bla') 

In [ ]:
IPython.display.Code(beautify(content))

Beware of the version field format

In [ ]:
d = {
    'title': page['title'],
    'id': page['id'],
    'type': page['type'],
    'version': { 'number': version, 'notifyEdit': False, 'message': change_message },
    'body': {
        'storage': {
            'representation': 'storage',
            'value': content
        }
    }
}

json.dumps(d)

## Publish the updated content

*HTTPError: 409 Client Error* indicates, either version or content are invalid

In [ ]:
url = page['_links']['self']

put_response = requests.put(url, headers={'Content-Type': 'application/json'}, data=json.dumps(d), auth=authentication)

# error reason is hidden in response object: therefore dump it
if not put_response.status_code == requests.codes.ok:
    from pprint import pprint
    from urllib.error import HTTPError
    pprint(vars(put_response))
    raise HTTPError(url, put_response.status_code, put_response.json(), put_response.headers, None)

